# Convert raw to dsToolbox format

The `ds-Toolbox` wants input in the form of a specific csv-file pr charge or discharge pr temperature.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

import cellpy
from cellpy.utils import plotutils

cellpy.__version__

In [ ]:
cur_dir = Path(".")
raw_data_path = cur_dir / "raw" / "combined_protocol_results_realistic.bdf.csv"
assert raw_data_path.exists()

In [ ]:
c = cellpy.get(
    raw_data_path,
    instrument="batmo_bdf",
    cycle_mode="full_cell",
    mass=1.0,
    nominal_capacity=120.0,
    nom_cap_specifics="absolute",
    refuse_copying=True,
)

In [ ]:
c.data.raw.head()

In [ ]:
# What are the units?
c.data.raw_units

In [ ]:
# We have a custom column that we want to also include in the summary (pr. cycle statistics) to find the cycles we want to export
c.add_to_summary("Maximum Cell Temperature / degC");

## Some information about the format that `ds-Toolbox` likes

We need to create individual .csv files for each charge and each discharge half-cycle. The cycles we want are the slow ones (characterization cycles), typically called an RPT. We know (maybe, if we had time to go through it in the cellpy session) that there were four characterization cycles in the beginning and then three at the end. Each RPT consists of one cycle at 15, 25, 35, and 45 deg C, all at C/20 rate. A rest period before and after each half-cycle also has to be included in each file.

We could in principle search for the cycles using code, but today we will look at the summary plots and voltage curves to figure out what we want to have.


In [ ]:
plotutils.summary_plot(c.from_cycle(2), hover_columns=["Maximum Cell Temperature / degC"], formation_cycles=None)

In [ ]:
plotutils.summary_plot(c, y="capacities_absolute_with_rate", interactive=True, filters={"rate": (0.49, 0.51)}, hover_columns=["Maximum Cell Temperature / degC"], formation_cycles=None)

## Export the first RPT

### First cycle (15 deg C)

In [ ]:
# We have to split into charge and discharge. Lets look at the data and find which ones.
plotutils.cycle_info_plot(c, cycle=2)

In [ ]:
# Helper function that we will need later when exporting (resetting step numbers, selecting step numbers, reversing the current)

from functools import partial

def tweak_something(df: pd.DataFrame, steps=None) -> pd.DataFrame:
    if steps is not None:
        df = df.query("step_index in @steps")
    df["step_index"] = (
    df.groupby("cycle_index")["step_index"]
      .transform(lambda s: pd.factorize(s)[0] + 1)
      .astype("int64")
)
    df.current = -df.current
    return df

In [ ]:
# if we want to change from standard bdf units, we need to provide a CellpyUnits object with wanted units. 

from cellpy.parameters.internal_settings import CellpyUnits
bdf_units = CellpyUnits(time="hrs")

In [ ]:
# Where to save the files

outdir = cur_dir / "out"
outdir.mkdir(exist_ok=True)

In [ ]:
c.to_bdf(outdir / "BattMo_OCV_P15_S3.csv", cycles=[2], extras=True, preprocess_fn=partial(tweak_something, steps=[6, 7, 8]), bdf_units=bdf_units)

In [ ]:
c.to_bdf(outdir / "BattMo_OCV_P15_S1.csv", cycles=[2], extras=True, preprocess_fn=partial(tweak_something, steps=[8, 9, 10]), bdf_units=bdf_units)

### Create the files for 25, 35, and 45 deg C

OK, this was a bit tedious - lets do the rest in one go...

In [ ]:
# Creating a small script that does it all in one go...

from functools import partial
 
#- 15degC_S3, cycle 2, step_index [1,2,3]
#- 15degC_S1, cycle 2, step_index [3,4,5]
#- 25degC_S3, cycle 3, step_index [1,2,3]
#- 25degC_S1, cycle 3, step_index [3,4,5]
#- 35degC_S3, cycle 4, step_index [1,2,3]
#- 35degC_S1, cycle 4, step_index [3,4,5]
#- 45degC_S3, cycle 5, step_index [1,2,3]
#- 45degC_S1, cycle 5, step_index [3,4,5]

bdf_lookup = {
    # We already did it for 15 deg C. So commented those out...
    # 15: {
    #     "charge": {
    #         "filename": "BattMo_OCV_P15_S3.csv",
    #         "cycles": [2],
    #         "steps": [6, 7, 8],
    #     },
    #     "discharge": {
    #         "filename": "BattMo_OCV_P15_S1.csv",
    #         "cycles": [2],
    #         "steps": [8, 9, 10],
    #     },
    # },
    25: {
        "charge": {
            "filename": "BattMo_OCV_P25_S3.csv",
            "cycles": [3],
            "steps": [11, 12, 13],

        },
        "discharge": {
            "filename": "BattMo_OCV_P25_S1.csv",
            "cycles": [3],
            "steps": [13, 14, 15],
        },
    },
    35: {
        "charge": {
            "filename": "BattMo_OCV_P35_S3.csv",
            "cycles": [4],
            "steps": [16, 17, 18],   
        },
        "discharge": {
            "filename": "BattMo_OCV_P35_S1.csv",
            "cycles": [4],
            "steps": [18, 19, 20],  
        },
    },
    45: {
        "charge": {
            "filename": "BattMo_OCV_P45_S3.csv",
            "cycles": [5],
            "steps": [21, 22, 23],   
        },
        "discharge": {
            "filename": "BattMo_OCV_P45_S1.csv",
            "cycles": [5],
            "steps": [23, 24, 25],  
        },
    },
}

# Looping through all temperatures
for temp, modes in bdf_lookup.items():
    for mode, cfg in modes.items():
        c.to_bdf(
            outdir / cfg["filename"],
            cycles=cfg["cycles"],
            extras=True,
            preprocess_fn=partial(tweak_something, steps=cfg["steps"]),
            bdf_units=bdf_units,
        )
 

## Export the last RPT

In [ ]:
# Home excercise ;-)